# 08 Multimodal Feature Experiments

This notebook investigates whether more specific image–text comparisons help detect manipulation. It builds two feature groups from the processed data in `02` and the saved CLIP outputs in `05`:

| Feature group | Comparison | Columns |
|---|---|---:|
| `mm_phrase` | Caption phrases against the whole image | 14 |
| `mm_name_face` | Person names against individual faces | 7 |

Each row represents an image–caption pair, identified by `row_id`. The features measure similarity, counts and availability. A similarity score of 0.2 does not mean a 20% probability of manipulation; the classifiers learn that prediction from labelled training examples.

The experiments use fixed logistic-regression settings to compare each group with an eight-score CLIP baseline and its count control. Images and faces reuse their saved embeddings; only the new text inputs need encoding. Features from `03` and `04` are not required here.

Models are trained on `train_sample`; all reported scores use the validation set. Feature files are exported for each split using the same calculations. Numerical examples in the feature descriptions illustrate the calculations rather than report experimental results.


## 0. Setup

Run from `notebooks/` with the project kernel. The settings are seed `4012`, up to eight phrases per category, and 500 article-level bootstrap resamples.

File and configuration hashes identify matching caches and record the inputs used for a run. JSON files are written through a temporary file to avoid leaving a partially written cache.


In [19]:
import os
import sys
import time
import json
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from src.evaluation import evaluate, best_f1_threshold, combine_fake_score, detection_rate_by_type

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FEATURES_DIR = PROJECT_ROOT / "data" / "features"
CACHE_DIR = PROJECT_ROOT / "data" / "interim" / "phrase_clip"
FEATURES_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

SPLITS = ["train_sample", "validation", "test"]
RANDOM_STATE = 4012
MAX_PHRASES = 8
CLIP_COLUMNS = ["clip_sim", "clip_sim_txt_norm", "clip_sim_img_norm", "clip_sim_txt_rank"]

sns.set_theme(style="whitegrid")

from IPython.display import Markdown, display
from src.evaluation import binary_metrics

FACE_CACHE = PROJECT_ROOT / "data/interim/clip"
EXPERIMENT_CACHE = PROJECT_ROOT / "data/interim/multimodal_07"
EXPERIMENT_CACHE.mkdir(parents=True, exist_ok=True)
TARGETS = ["image_fake", "text_fake", "is_fake"]
FACE_COLUMNS = ["clip_face_sim_max", "clip_face_sim_min", "clip_face_sim_mean", "clip_face_sim_largest"]
BOOTSTRAPS = 500


def fingerprint(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False).encode()).hexdigest()


def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(".tmp")
    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False))
    temporary.replace(path)


timings = {}


## 1. Load Data and CLIP Outputs

Processed metadata and the `clip`, `clip_emb` and `clip_face` tables are joined by `row_id`. Their IDs must be unique and cover the same pairs. Embeddings are reordered to match the metadata, and training and validation articles are checked for overlap.

| Input from `05` | Use in this notebook |
|---|---|
| `clip_emb_<split>.parquet` | 768-dimensional image and caption vectors from ViT-L/14. Image vectors are converted to float32 and L2-normalised. A few normalised caption vectors are retained as encoder anchors. |
| `clip_<split>.parquet` | Four existing global similarity features, described below. |
| `clip_face_<split>.parquet` | Four existing face–whole-caption similarity summaries. Missing summaries are filled with zero here; the name–face group includes face availability/count controls. |
| Raw ViT-B/16 face/caption caches | Loaded for name–face work in section 3. They use 512 dimensions and must not be compared directly with ViT-L/14 vectors. |

The baseline includes four whole-image–caption scores calculated in `05`. Let `c = clip_sim` for the paired image and caption:

| Column | Calculation made in `05` |
|---|---|
| `clip_sim` | Cosine similarity between the paired image and caption embeddings. |
| `clip_sim_txt_norm` | `c` minus the caption's mean similarity to a reference set of training images. |
| `clip_sim_img_norm` | `c` minus the image's mean similarity to a reference set of training captions. |
| `clip_sim_txt_rank` | Fraction of reference images whose similarity with this caption is **strictly lower** than `c`. |

`05` samples up to 2,000 reference images/captions from training data. Four ViT-B/16 face–caption scores complete the eight-feature baseline: `clip_face_sim_max`, `clip_face_sim_min`, `clip_face_sim_mean` and `clip_face_sim_largest`. These are maximum/minimum/mean across faces and the score for the face with greatest bounding-box area. They compare faces with the **whole caption**, unlike the new name-specific comparisons.

Each split now has a table containing metadata, labels and the existing scores. The experiment definitions select the predictor columns explicitly, excluding labels, IDs and paths.


In [20]:
data = {}
image_vectors = {}
caption_anchors = []

for split in SPLITS:
    rows = pd.read_parquet(PROCESSED_DIR / f"{split}.parquet")
    clip = pd.read_parquet(FEATURES_DIR / f"clip_{split}.parquet")
    emb = pd.read_parquet(FEATURES_DIR / f"clip_emb_{split}.parquet")
    assert rows.row_id.is_unique and clip.row_id.is_unique and emb.row_id.is_unique
    assert set(rows.row_id) == set(clip.row_id) == set(emb.row_id), f"row IDs differ in {split}"
    rows = rows.merge(clip, on="row_id", how="left", validate="one_to_one", sort=False)
    emb = emb.set_index("row_id").loc[rows.row_id]
    img_cols = sorted(c for c in emb if c.startswith("clip_img_"))
    txt_cols = sorted(c for c in emb if c.startswith("clip_txt_"))
    assert len(img_cols) == len(txt_cols) == 768, "Expected ViT-L/14 embeddings from 05"
    image = emb[img_cols].to_numpy(dtype=np.float32, copy=True)
    norms = np.linalg.norm(image, axis=1, keepdims=True)
    assert np.isfinite(image).all() and (norms > 0).all()
    image_vectors[split] = image / norms
    assert np.isfinite(rows[CLIP_COLUMNS].to_numpy()).all()
    for i in range(min(3, len(rows))):
        anchor = emb[txt_cols].iloc[i].to_numpy(dtype=np.float32, copy=True)
        assert np.isfinite(anchor).all() and np.linalg.norm(anchor) > 0
        caption_anchors.append((rows.text.iloc[i], anchor / np.linalg.norm(anchor)))
    data[split] = rows

assert not (set(data["train_sample"].id) & set(data["validation"].id)), "Article overlap"
all_rows = pd.concat(data.values(), ignore_index=True)
captions = pd.Series(all_rows.text.unique(), name="text")
pd.Series({split: len(df) for split, df in data.items()}, name="rows")

for split, rows in data.items():
    face_scores = pd.read_parquet(FEATURES_DIR / f"clip_face_{split}.parquet")
    assert face_scores.row_id.is_unique and set(face_scores.row_id) == set(rows.row_id)
    data[split] = rows.merge(face_scores[["row_id"] + FACE_COLUMNS], on="row_id", validate="one_to_one", sort=False)
    # No-face summaries from 05 can be NaN: the later availability controls distinguish them.
    data[split][FACE_COLUMNS] = data[split][FACE_COLUMNS].fillna(0.)
    assert np.isfinite(data[split][FACE_COLUMNS].to_numpy()).all()

input_files = [PROCESSED_DIR / f"{s}.parquet" for s in SPLITS]
input_files += [FEATURES_DIR / f"{g}_{s}.parquet" for g in ["clip", "clip_emb", "clip_face"] for s in SPLITS]
data_fingerprints = {str(p.relative_to(PROJECT_ROOT)): file_sha256(p) for p in input_files}


## 2. Phrase–Image Consistency

**Hypothesis:** an individual activity or setting can disagree with the image even when the whole caption seems plausible. This is measured against the whole image, not a detected local region.

### Extracting phrases

- **Nouns:** use spaCy `doc.noun_chunks`, remove exact duplicate strings while preserving order, and retain the first eight. There is no case-folding in this noun/verb deduplication.
- **Verbs:** for each token tagged `VERB`, select its position plus the subtrees of children with dependency labels `aux`, `auxpass`, `neg`, `prt`, `advmod`, `obj`, `dobj`, `attr` or `prep`. Take the contiguous span from the earliest to latest selected position; it may contain intervening tokens. Discard spans longer than 20 tokens, remove exact duplicates, and retain the first eight.
- Named-entity recognition is disabled for phrase extraction. Names can still appear in noun chunks; separate `PERSON` extraction is covered in section 3. The sample phrase lists help identify parsing errors.

### Phrase features

For one category, let `n` be the number of retained phrases, `s_1 … s_n` their image–phrase similarities, `m = mean(s)`, and `c` the stored whole-caption `clip_sim`. Each `s_i` is the dot product of the normalised image and CLIP text vector (cosine similarity).

| Noun column | Verb column | Exact calculation / interpretation |
|---|---|---|
| `mm_phrase_noun_count` | `mm_phrase_verb_count` | `n`, between 0 and 8. This counts retained phrases, not all words in the caption. |
| `mm_phrase_noun_missing` | `mm_phrase_verb_missing` | `1` if `n == 0`, otherwise `0`. |
| `mm_phrase_noun_min` | `mm_phrase_verb_min` | `min(s)`: weakest-matching retained phrase. |
| `mm_phrase_noun_max` | `mm_phrase_verb_max` | `max(s)`: strongest-matching retained phrase. |
| `mm_phrase_noun_mean` | `mm_phrase_verb_mean` | `sum(s) / n`: average agreement. |
| `mm_phrase_noun_std` | `mm_phrase_verb_std` | `sqrt(mean((s - m)**2))`: population standard deviation, matching NumPy's default `std(ddof=0)`. It is 0 for one phrase. |
| `mm_phrase_noun_caption_gap` | `mm_phrase_verb_caption_gap` | `c - m`: how much more/less the whole caption matches than the average phrase. A positive gap does not by itself establish an unsupported detail. |

**Missing phrases:** when `n=0`, count is 0, missing is 1, and all five numerical similarity/gap summaries are 0. In particular, `caption_gap` is forced to 0; the code does not calculate `c - 0` for missing phrases.

**Example:** similarities `[0.10, 0.20, 0.30]` and `clip_sim=0.25` give count `3`, missing `0`, minimum `0.10`, maximum `0.30`, mean `0.20`, standard deviation approximately `0.08165`, and caption gap `0.05`.

The two categories produce **14 columns: ten similarity/gap measurements and four count/missing indicators**. Phrase length and context outside the image can affect these scores even for genuine pairs.


In [21]:
def phrases_from_doc(doc):
    nouns = list(dict.fromkeys(chunk.text for chunk in doc.noun_chunks))[:MAX_PHRASES]
    verbs = []
    for token in doc:
        if token.pos_ != "VERB":
            continue
        positions = [token.i]
        for child in token.children:
            if child.dep_ in {"aux", "auxpass", "neg", "prt", "advmod", "obj", "dobj", "attr", "prep"}:
                positions.extend(word.i for word in child.subtree)
        lo, hi = min(positions), max(positions) + 1
        if hi - lo <= 20:
            verbs.append(doc[lo:hi].text)
    return {"noun": nouns, "verb": list(dict.fromkeys(verbs))[:MAX_PHRASES]}


def build_phrase_features(rows, image_vectors, phrase_lookup, phrase_vectors):
    records = []
    for i, row in enumerate(rows.itertuples()):
        record = {"row_id": row.row_id}
        for category in ["noun", "verb"]:
            phrases = phrase_lookup[row.text][category]
            similarities = np.asarray([image_vectors[i] @ phrase_vectors[p] for p in phrases])
            prefix = f"mm_phrase_{category}_"
            record[prefix + "count"] = len(phrases)
            record[prefix + "missing"] = int(not phrases)
            stats = ({"min": similarities.min(), "max": similarities.max(),
                      "mean": similarities.mean(), "std": similarities.std()} if phrases else
                     dict.fromkeys(["min", "max", "mean", "std"], 0.))
            record.update({prefix + name: float(value) for name, value in stats.items()})
            record[prefix + "caption_gap"] = row.clip_sim - stats["mean"] if phrases else 0.
        records.append(record)
    return pd.DataFrame(records)


Each distinct caption is parsed once. Repeated captions reuse the same phrase list, while similarity calculations use the image paired with each row. Parsed phrases are cached using the caption order, phrase limit, spaCy/model metadata and extraction method. Sample phrase lists are shown for inspection.


In [22]:
phrase_stage_start = time.perf_counter()
import spacy

nlp = spacy.load("en_core_web_sm", disable=["lemmatizer", "ner"])
phrase_config = {"captions": captions.tolist(), "max_phrases": MAX_PHRASES,
                 "spacy": spacy.__version__, "model": nlp.meta, "method": "noun_chunks_verb_subtrees_v1"}
phrase_key = hashlib.sha256(json.dumps(phrase_config, sort_keys=True).encode()).hexdigest()[:16]
phrase_path = CACHE_DIR / f"caption_phrases_{phrase_key}.json"

if phrase_path.exists():
    phrase_lookup = json.loads(phrase_path.read_text())
    print("Caption phrases loaded from cache")
else:
    start = time.time()
    phrase_lookup = {text: phrases_from_doc(doc)
                     for text, doc in zip(captions, nlp.pipe(captions, batch_size=128))}
    temporary = phrase_path.with_suffix(".tmp")
    temporary.write_text(json.dumps(phrase_lookup))
    temporary.replace(phrase_path)
    print(f"spaCy phrases: {time.time() - start:.0f}s")

assert set(phrase_lookup) == set(captions)
pd.DataFrame([{"text": text, **phrase_lookup[text]} for text in captions.head(5)])


Caption phrases loaded from cache


,text,noun,verb
0,Chanan Reitblat told the court said felt viola...,"[Chanan Reitblat, the court]","[told, said, felt violated, devastated]"
1,David Oyelowo left and Michael Keaton,"[David Oyelowo, Michael Keaton]",[left]
2,No contest LeRoy Neiman presents the boxer Ber...,"[No contest, LeRoy Neiman, the boxer, Bernard ...",[presents the boxer Bernard Hopkins with a por...
3,Rev Jesse Jackson greets attendees before the ...,"[Rev Jesse Jackson greets, the start, the vice...",[]
4,Gingrich talks to Jackson and his father durin...,"[Gingrich talks, Jackson, his father, a break,...",[]


### 2.1 Phrase Embeddings

Unique phrases are encoded with frozen **ViT-L/14**, matching the whole-image embeddings from `05`. `encode_texts` returns normalised vectors, so their dot products with normalised image vectors give cosine similarities.

Encoding uses batches of 64, with checkpoints every 1,024 text inputs. Cache keys include text order, model revision, library versions, device and batch size. Loaded chunks are checked for the expected shape and finite values.

A few whole captions provide reference vectors for checking compatibility with `05`. Re-encoded vectors must match within absolute tolerance `5e-4`, with zero relative tolerance. This check covers those reference captions. The model is released from memory after encoding.


In [23]:
import torch
import transformers
from src.clip_features import CLIP_NAME, best_device, load_clip, encode_texts

phrases = list(dict.fromkeys(phrase for item in phrase_lookup.values()
                            for category in ["noun", "verb"] for phrase in item[category]))
prompts = list(dict.fromkeys(phrases + [text for text, _ in caption_anchors]))
device = best_device()
model, _, tokenizer = load_clip(device, CLIP_NAME)
encoder_config = {"model": CLIP_NAME, "revision": getattr(model.config, "_commit_hash", None),
                  "torch": str(torch.__version__), "transformers": transformers.__version__,
                  "device": device, "batch_size": 64, "prompts": prompts}
encoder_key = hashlib.sha256(json.dumps(encoder_config, sort_keys=True).encode()).hexdigest()[:16]
embedding_dir = CACHE_DIR / encoder_key
embedding_dir.mkdir(parents=True, exist_ok=True)

parts = []
start = time.time()
for lo in range(0, len(prompts), 1024):
    batch = prompts[lo:lo + 1024]
    path = embedding_dir / f"phrases_{lo:07d}.npy"
    if path.exists():
        part = np.load(path, allow_pickle=False)
    else:
        part = encode_texts(model, tokenizer, batch, device, batch_size=64)
        temporary = path.with_suffix(".tmp")
        with temporary.open("wb") as stream:
            np.save(stream, part, allow_pickle=False)
        temporary.replace(path)
    assert part.shape == (len(batch), 768) and np.isfinite(part).all()
    parts.append(part)
    print(f"{min(lo + 1024, len(prompts)):,}/{len(prompts):,} prompts; {time.time() - start:.0f}s", flush=True)

prompt_emb = np.concatenate(parts)
phrase_vectors = dict(zip(prompts, prompt_emb))
for text, expected in caption_anchors:
    assert np.allclose(phrase_vectors[text], expected, atol=5e-4, rtol=0), "Text encoder differs from 05"
print(f"{len(phrases):,} distinct phrases; {CLIP_NAME} on {device}")
del model


Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

1,024/179,289 prompts; 0s
2,048/179,289 prompts; 0s
3,072/179,289 prompts; 0s
4,096/179,289 prompts; 0s
5,120/179,289 prompts; 0s
6,144/179,289 prompts; 0s
7,168/179,289 prompts; 0s
8,192/179,289 prompts; 0s
9,216/179,289 prompts; 0s
10,240/179,289 prompts; 0s
11,264/179,289 prompts; 0s
12,288/179,289 prompts; 0s
13,312/179,289 prompts; 0s
14,336/179,289 prompts; 0s
15,360/179,289 prompts; 0s
16,384/179,289 prompts; 0s
17,408/179,289 prompts; 0s
18,432/179,289 prompts; 0s
19,456/179,289 prompts; 0s
20,480/179,289 prompts; 0s
21,504/179,289 prompts; 0s
22,528/179,289 prompts; 0s
23,552/179,289 prompts; 0s
24,576/179,289 prompts; 0s
25,600/179,289 prompts; 0s
26,624/179,289 prompts; 0s
27,648/179,289 prompts; 0s
28,672/179,289 prompts; 0s
29,696/179,289 prompts; 0s
30,720/179,289 prompts; 0s
31,744/179,289 prompts; 0s
32,768/179,289 prompts; 0s
33,792/179,289 prompts; 0s
34,816/179,289 prompts; 0s
35,840/179,289 prompts; 0s
36,864/179,289 prompts; 0s
37,888/179,289 prompts; 0s
38,912/179

`build_phrase_features` compares each caption's phrases with its paired image and calculates the 14 columns defined above. Row IDs must be unique across splits, and all feature values must be finite. `describe()` summarises the feature distributions.


In [24]:
features = pd.concat([build_phrase_features(rows, image_vectors[split], phrase_lookup, phrase_vectors)
                      for split, rows in data.items()], ignore_index=True)
assert features.row_id.is_unique
assert np.isfinite(features.drop(columns="row_id").to_numpy()).all()
feature_columns = [c for c in features if c != "row_id"]
print(f"{len(feature_columns)} features for {len(features):,} image-caption pairs")
features[feature_columns].describe().round(3)


14 features for 122,774 image-caption pairs


,mm_phrase_noun_count,mm_phrase_noun_missing,mm_phrase_noun_min,mm_phrase_noun_max,mm_phrase_noun_mean,mm_phrase_noun_std,mm_phrase_noun_caption_gap,mm_phrase_verb_count,mm_phrase_verb_missing,mm_phrase_verb_min,mm_phrase_verb_max,mm_phrase_verb_mean,mm_phrase_verb_std,mm_phrase_verb_caption_gap
count,122774.000,122774.000,122774.000,122774.000,122774.000,122774.000,122774.000,122774.000,122774.000,122774.000,122774.000,122774.000,122774.000,122774.000
mean,4.732,0.000,0.133,0.230,0.176,0.036,0.079,1.688,0.047,0.170,0.200,0.184,0.014,0.059
std,1.695,0.012,0.034,0.039,0.028,0.016,0.050,0.992,0.212,0.066,0.068,0.064,0.020,0.055
min,0.000,0.000,-0.110,0.000,0.000,0.000,-0.234,0.000,0.000,-0.052,-0.041,-0.041,0.000,-0.210
25%,3.000,0.000,0.114,0.203,0.158,0.025,0.050,1.000,0.000,0.132,0.164,0.153,0.000,0.016
50%,5.000,0.000,0.132,0.231,0.175,0.035,0.082,1.000,0.000,0.162,0.205,0.185,0.000,0.057
75%,6.000,0.000,0.151,0.256,0.194,0.047,0.112,2.000,0.000,0.213,0.246,0.223,0.024,0.097
max,8.000,1.000,0.385,0.422,0.385,0.142,0.269,8.000,1.000,0.433,0.433,0.433,0.149,0.303


Phrase features are saved as `mm_phrase_<split>.parquet`, with `row_id` and 14 feature columns in the original split order. Labels and raw embeddings are excluded. Large temporary arrays are released before loading the face/text model. Runtime includes cache reads, so a cached run does not measure the cost of extracting everything from scratch.


In [25]:
def save_features(features, group):
    for split, rows in data.items():
        part = rows[["row_id"]].merge(features, on="row_id", how="left", validate="one_to_one", sort=False)
        assert part.row_id.tolist() == rows.row_id.tolist()
        assert np.isfinite(part.drop(columns="row_id").to_numpy()).all()
        path = FEATURES_DIR / f"{group}_{split}.parquet"
        temporary = path.with_suffix(".tmp.parquet")
        part.to_parquet(temporary, index=False)
        temporary.replace(path)
        print(f"{path.relative_to(PROJECT_ROOT)}  {part.shape}")


save_features(features, "mm_phrase")

phrase_features = features.copy()
phrase_columns = feature_columns.copy()
timings["phrase_stage_seconds_including_cache_reads"] = time.perf_counter() - phrase_stage_start
# Release large temporary arrays before loading the smaller face/text model.
del image_vectors, phrase_vectors, prompt_emb, parts, part, emb, image
import gc
gc.collect()
if torch.backends.mps.is_available():
    torch.mps.empty_cache()


data/features/mm_phrase_train_sample.parquet  (49943, 15)
data/features/mm_phrase_validation.parquet  (22126, 15)
data/features/mm_phrase_test.parquet  (50705, 15)


## 3. Person-Name–Face Consistency

Face embeddings from `05` are mapped back to images using the same distinct-image and bounding-box order. Images may have multiple faces, and repeated images reuse their vectors. Array sizes and values are checked, and the ordered mapping is saved for cache reuse.


In [26]:
def build_face_index(rows):
    images = rows.drop_duplicates("image")[["image", "mtcnn_boxes"]].reset_index(drop=True)
    counts = images.mtcnn_boxes.str.len().to_numpy(dtype=int)
    owner = np.repeat(np.arange(len(images)), counts)
    boxes = np.asarray([b for items in images.mtcnn_boxes for b in items], dtype=float).reshape(-1, 4)
    assert np.isfinite(boxes).all() and ((boxes[:, 2:] - boxes[:, :2]) > 0).all()
    manifest = pd.DataFrame(boxes, columns=["x1", "y1", "x2", "y2"])
    manifest.insert(0, "image", images.image.to_numpy()[owner])
    manifest.insert(0, "face_index", np.arange(len(owner)))
    starts = np.r_[0, np.cumsum(counts)[:-1]]
    return images, counts, starts, manifest




start = time.perf_counter()
all_rows = pd.concat(data.values(), ignore_index=True)
images, face_counts, face_starts, face_manifest = build_face_index(all_rows)
face_emb = np.load(FACE_CACHE / "faces_vitb16.npy", mmap_mode="r", allow_pickle=False)
face_caption_emb = np.load(FACE_CACHE / "captions_vitb16.npy", mmap_mode="r", allow_pickle=False)
assert face_emb.shape == (int(face_counts.sum()), 512)
assert face_caption_emb.shape == (len(captions), 512)
for array in [face_emb, face_caption_emb]:
    for lo in range(0, len(array), 20000):
        block = array[lo:lo + 20000]
        assert np.isfinite(block).all() and np.allclose(np.linalg.norm(block, axis=1), 1., atol=1e-4)

face_identity = {
    "schema": 1, "model": "openai/clip-vit-base-patch16", "margin": 0.2,
    "face_file": file_sha256(FACE_CACHE / "faces_vitb16.npy"),
    "caption_file": file_sha256(FACE_CACHE / "captions_vitb16.npy"),
    "face_order": hashlib.sha256(pd.util.hash_pandas_object(face_manifest, index=False).values.tobytes()).hexdigest(),
    "caption_order": fingerprint(captions.tolist()),
    "row_order": fingerprint(all_rows.row_id.tolist()),
    "reference_scores": {s: data_fingerprints[f"data/features/clip_face_{s}.parquet"] for s in SPLITS},
}
manifest_path = EXPERIMENT_CACHE / "verified_face_manifest.json"
index_path = EXPERIMENT_CACHE / "verified_face_index.parquet"
if manifest_path.exists():
    assert json.loads(manifest_path.read_text()) == face_identity, "Face cache identity changed; verify its provenance"
    assert index_path.exists() and pd.read_parquet(index_path).equals(face_manifest), "Face-index manifest differs"

if not manifest_path.exists():
    temporary = index_path.with_suffix(".tmp.parquet")
    face_manifest.to_parquet(temporary, index=False)
    temporary.replace(index_path)
    write_json(manifest_path, face_identity)
timings["face_alignment_seconds"] = time.perf_counter() - start
row_image_index = pd.Index(images.image).get_indexer(all_rows.image)
print(f"Loaded {len(face_emb):,} cached faces and mapped them to {len(all_rows):,} image-caption pairs")


Loaded 294,383 cached faces and mapped them to 122,774 image-caption pairs


### 3.1 Person Names and Matching Features

**Hypothesis:** isolating names may reveal identity-related disagreement obscured when a face is compared with the whole caption.

spaCy extracts `PERSON` entities once per distinct caption. Internal whitespace is normalised, and names differing only in case are merged while keeping the first spelling. There is no eight-name limit. Extraction is checkpointed every 2,048 captions. These are automatically extracted names, so omissions and incorrect entities are possible.

Each unique name becomes `a photo of {name}` and is encoded with ViT-B/16 in section 3.2. For a pair with `N` names and `F` faces, let `S[j,k]` be cosine similarity between name prompt `j` and face `k`. The best match for name `j` is `b_j = max_k S[j,k]`.

| Exported column | Exact calculation |
|---|---|
| `mm_name_face_name_count` | `N`, the number of deduplicated extracted names. |
| `mm_name_face_face_count` | `F`, the number of cached faces for the paired image. |
| `mm_name_face_available` | `1` when both `N>0` and `F>0`; otherwise `0`. |
| `mm_name_face_gap_available` | `1` when matching is available **and `F>=2`**. It requires two faces, not two names. |
| `mm_name_face_best_mean` | Mean of the `b_j` values: average best-face match across names. |
| `mm_name_face_best_min` | Minimum of the `b_j` values: the weakest name's best match. This is not the weakest entry of the whole similarity matrix. |
| `mm_name_face_face_gap_mean` | For each name, take its highest minus second-highest **face** similarity; average those gaps across names. |

**Missing names or faces:** if names or faces are absent, all three matching summaries stay 0 and availability is 0; counts remain actual counts. With one face, best-mean/minimum are still calculated, but gap is 0 and gap-availability is 0.

**Example similarity matrix** (names in rows, faces in columns):

| Prompt | Face 1 | Face 2 | Best match | Best minus second-best face |
|---|---:|---:|---:|---:|
| `a photo of Alice` | 0.30 | 0.10 | 0.30 | 0.20 |
| `a photo of Bob` | 0.12 | 0.28 | 0.28 | 0.16 |

The exported best-mean is `0.29`, best-minimum `0.28`, and mean face gap `0.18`. Both counts are 2 and both indicators are 1.

The group contains **seven columns: three matching measurements and four count/availability controls**. Names are matched independently, so two names can select the same face. A larger gap indicates a stronger preference between faces, rather than verified identity. Genuine news can also mention unfamiliar people or people absent from the image.


In [27]:
def names_from_doc(doc):
    unique = {}
    for entity in doc.ents:
        if entity.label_ == "PERSON":
            name = " ".join(entity.text.split())
            if name:
                unique.setdefault(name.casefold(), name)
    return list(unique.values())


start = time.perf_counter()
name_nlp = spacy.load("en_core_web_sm", disable=["parser", "lemmatizer"])
name_config = {"captions": captions.tolist(), "spacy": spacy.__version__, "model": name_nlp.meta,
               "method": "person_entities_whitespace_casefold_v1"}
name_key = fingerprint(name_config)
name_dir = EXPERIMENT_CACHE / "names" / name_key[:16]
name_dir.mkdir(parents=True, exist_ok=True)
name_lookup = {}
for lo in range(0, len(captions), 2048):
    batch = captions.iloc[lo:lo + 2048].tolist()
    path = name_dir / f"names_{lo:07d}.json"
    if path.exists():
        chunk = json.loads(path.read_text())
    else:
        chunk = {text: names_from_doc(doc) for text, doc in zip(batch, name_nlp.pipe(batch, batch_size=128))}
        write_json(path, chunk)
    assert list(chunk) == batch
    assert all(isinstance(names, list) and all(isinstance(n, str) and n for n in names)
               and len({n.casefold() for n in names}) == len(names) for names in chunk.values())
    name_lookup.update(chunk)
    print(f"Names: {min(lo + 2048, len(captions)):,}/{len(captions):,} captions", flush=True)
del name_nlp
timings["name_extraction_seconds"] = time.perf_counter() - start
pd.Series([len(name_lookup[text]) for text in captions]).describe().rename("names per unique caption")


Names: 2,048/54,588 captions
Names: 4,096/54,588 captions
Names: 6,144/54,588 captions
Names: 8,192/54,588 captions
Names: 10,240/54,588 captions
Names: 12,288/54,588 captions
Names: 14,336/54,588 captions
Names: 16,384/54,588 captions
Names: 18,432/54,588 captions
Names: 20,480/54,588 captions
Names: 22,528/54,588 captions
Names: 24,576/54,588 captions
Names: 26,624/54,588 captions
Names: 28,672/54,588 captions
Names: 30,720/54,588 captions
Names: 32,768/54,588 captions
Names: 34,816/54,588 captions
Names: 36,864/54,588 captions
Names: 38,912/54,588 captions
Names: 40,960/54,588 captions
Names: 43,008/54,588 captions
Names: 45,056/54,588 captions
Names: 47,104/54,588 captions
Names: 49,152/54,588 captions
Names: 51,200/54,588 captions
Names: 53,248/54,588 captions
Names: 54,588/54,588 captions


count    54588.000000
mean         1.304847
std          0.983421
min          0.000000
25%          1.000000
50%          1.000000
75%          2.000000
max          8.000000
Name: names per unique caption, dtype: float64

### 3.2 Name Embeddings

Each unique `a photo of {name}` string is encoded with ViT-B/16, matching the cached face vectors. `face_prompt_vectors` stores the 512-dimensional normalised vector for each string. A few whole captions are included as reference checks against `05`.

Encoding uses batches of 64 and checkpoints of 1,024 text inputs. Cache identity includes the model revision, ordered inputs, device and library versions. Loaded vectors are checked for shape, finite values and approximately unit length. Reference caption vectors must match `05` within absolute tolerance `5e-4`, with zero relative tolerance. The model is released after encoding.


In [28]:
from src.clip_features import FACE_CLIP_NAME

name_prompts = list(dict.fromkeys(f"a photo of {name}" for names in name_lookup.values() for name in names))
caption_position = {text: i for i, text in enumerate(captions)}
face_anchors = [(text, face_caption_emb[caption_position[text]]) for text, _ in caption_anchors]
face_prompts = list(dict.fromkeys(name_prompts + [text for text, _ in face_anchors]))


def cached_text_chunks(model, tokenizer, prompts, device, directory, width=512, chunk_size=1024):
    parts = []
    directory.mkdir(parents=True, exist_ok=True)
    for lo in range(0, len(prompts), chunk_size):
        batch = prompts[lo:lo + chunk_size]
        path = directory / f"text_{lo:07d}.npy"
        if path.exists():
            part = np.load(path, allow_pickle=False)
        else:
            part = encode_texts(model, tokenizer, batch, device, batch_size=64)
            assert part.shape == (len(batch), width) and np.isfinite(part).all()
            temporary = path.with_suffix(".tmp")
            with temporary.open("wb") as stream:
                np.save(stream, part, allow_pickle=False)
            temporary.replace(path)
        assert part.shape == (len(batch), width) and np.isfinite(part).all(), "Invalid cached text chunk"
        assert np.allclose(np.linalg.norm(part, axis=1), 1., atol=1e-4)
        parts.append(part)
        print(f"Face-space prompts: {min(lo + chunk_size, len(prompts)):,}/{len(prompts):,}", flush=True)
    return np.concatenate(parts)


start = time.perf_counter()
face_model, _, face_tokenizer = load_clip(device, FACE_CLIP_NAME)
face_encoder_config = {"model": FACE_CLIP_NAME, "revision": getattr(face_model.config, "_commit_hash", None),
                       "torch": str(torch.__version__), "transformers": transformers.__version__,
                       "device": device, "batch_size": 64, "prompts": face_prompts}
face_encoder_key = fingerprint(face_encoder_config)
prompt_dir = EXPERIMENT_CACHE / "face_prompts" / face_encoder_key[:16]
face_prompt_emb = cached_text_chunks(face_model, face_tokenizer, face_prompts, device, prompt_dir)
face_prompt_vectors = dict(zip(face_prompts, face_prompt_emb))
for text, expected in face_anchors:
    assert np.allclose(face_prompt_vectors[text], expected, atol=5e-4, rtol=0), "ViT-B/16 encoder differs from 05"
del face_model
gc.collect()
if torch.backends.mps.is_available():
    torch.mps.empty_cache()
timings["name_prompt_encoding_seconds"] = time.perf_counter() - start
print(f"{len(name_prompts):,} unique name prompts; face-space anchors agree")


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Face-space prompts: 1,024/28,013
Face-space prompts: 2,048/28,013
Face-space prompts: 3,072/28,013
Face-space prompts: 4,096/28,013
Face-space prompts: 5,120/28,013
Face-space prompts: 6,144/28,013
Face-space prompts: 7,168/28,013
Face-space prompts: 8,192/28,013
Face-space prompts: 9,216/28,013
Face-space prompts: 10,240/28,013
Face-space prompts: 11,264/28,013
Face-space prompts: 12,288/28,013
Face-space prompts: 13,312/28,013
Face-space prompts: 14,336/28,013
Face-space prompts: 15,360/28,013
Face-space prompts: 16,384/28,013
Face-space prompts: 17,408/28,013
Face-space prompts: 18,432/28,013
Face-space prompts: 19,456/28,013
Face-space prompts: 20,480/28,013
Face-space prompts: 21,504/28,013
Face-space prompts: 22,528/28,013
Face-space prompts: 23,552/28,013
Face-space prompts: 24,576/28,013
Face-space prompts: 25,600/28,013
Face-space prompts: 26,624/28,013
Face-space prompts: 27,648/28,013
Face-space prompts: 28,013/28,013
28,004 unique name prompts; face-space anchors agree


### 3.3 Matching Names to Faces

`name_face_record` calculates the seven features in section 3.1 for each image–caption pair. Repeated images reuse face vectors; each caption supplies its own extracted names.

Results are checkpointed every 5,000 pairs. Cache keys include the ordered pair inputs, face-cache identity, name and encoder fingerprints, and method version. Cached chunks must have the expected columns, row order and finite values.

`mm_name_face_<split>.parquet` stores `row_id` and the seven features. The same label-free calculation is applied to every split.

The coverage table reports the fraction of pairs with usable inputs: at least one noun or verb phrase for phrases, and at least one name and face for names. `finite=True` confirms that the feature values are finite. Missing inputs remain identifiable through the availability indicators.


In [29]:
def name_face_record(names, faces, vectors):
    count, n_faces = len(names), len(faces)
    available = bool(count and n_faces)
    gap_available = available and n_faces >= 2
    out = {"name_count": count, "face_count": n_faces, "available": int(available),
           "gap_available": int(gap_available), "best_mean": 0., "best_min": 0., "face_gap_mean": 0.}
    if available:
        similarities = np.stack([vectors[f"a photo of {name}"] for name in names]) @ faces.T
        best = similarities.max(axis=1)
        out.update(best_mean=float(best.mean()), best_min=float(best.min()))
        if gap_available:
            top_two = np.partition(similarities, -2, axis=1)[:, -2:]
            out["face_gap_mean"] = float((top_two.max(axis=1) - top_two.min(axis=1)).mean())
    return {"mm_name_face_" + name: value for name, value in out.items()}


name_feature_config = {"method": "person_name_best_face_v1", "faces": face_identity,
                       "name_key": name_key, "encoder_key": face_encoder_key,
                       "rows": fingerprint(all_rows[["row_id", "text", "image"]].values.tolist())}
name_feature_key = fingerprint(name_feature_config)
name_feature_dir = EXPERIMENT_CACHE / "row_features" / name_feature_key[:16]
name_feature_dir.mkdir(parents=True, exist_ok=True)
name_columns = ["mm_name_face_" + suffix for suffix in
                ["name_count", "face_count", "available", "gap_available", "best_mean", "best_min", "face_gap_mean"]]
name_parts = []
start = time.perf_counter()
for lo in range(0, len(all_rows), 5000):
    batch = all_rows.iloc[lo:lo + 5000]
    name_path = name_feature_dir / f"names_{lo:07d}.parquet"
    if name_path.exists():
        names_part = pd.read_parquet(name_path)
    else:
        name_records = []
        for offset, row in enumerate(batch.itertuples()):
            image_index = row_image_index[lo + offset]
            first, count = face_starts[image_index], face_counts[image_index]
            stop = first + count
            name_records.append({"row_id": row.row_id, **name_face_record(
                name_lookup[row.text], face_emb[first:stop], face_prompt_vectors)})
        names_part = pd.DataFrame(name_records)
        temporary = name_path.with_suffix(".tmp.parquet")
        names_part.to_parquet(temporary, index=False)
        temporary.replace(name_path)
    assert names_part.columns.tolist() == ["row_id"] + name_columns
    assert names_part.row_id.is_unique and names_part.row_id.tolist() == batch.row_id.tolist()
    assert np.isfinite(names_part[name_columns].to_numpy()).all()
    name_parts.append(names_part)
    print(f"Name–face feature rows: {min(lo + 5000, len(all_rows)):,}/{len(all_rows):,}", flush=True)

name_features = pd.concat(name_parts, ignore_index=True)
assert name_features.row_id.is_unique
timings["name_face_summaries_seconds"] = time.perf_counter() - start
save_features(name_features, "mm_name_face")
feature_groups = {"phrases": phrase_features, "names": name_features}
coverage = []
for split, rows in data.items():
    for group, frame in feature_groups.items():
        part = frame.set_index("row_id").loc[rows.row_id]
        available = ((part.mm_phrase_noun_missing == 0) | (part.mm_phrase_verb_missing == 0)
                     if group == "phrases" else part.mm_name_face_available == 1)
        coverage.append({"split": split, "group": group, "rows": len(part), "features": len(part.columns),
                         "available_fraction": float(available.mean()), "finite": bool(np.isfinite(part.to_numpy()).all())})
coverage = pd.DataFrame(coverage)
display(coverage)


Name–face feature rows: 5,000/122,774
Name–face feature rows: 10,000/122,774
Name–face feature rows: 15,000/122,774
Name–face feature rows: 20,000/122,774
Name–face feature rows: 25,000/122,774
Name–face feature rows: 30,000/122,774
Name–face feature rows: 35,000/122,774
Name–face feature rows: 40,000/122,774
Name–face feature rows: 45,000/122,774
Name–face feature rows: 50,000/122,774
Name–face feature rows: 55,000/122,774
Name–face feature rows: 60,000/122,774
Name–face feature rows: 65,000/122,774
Name–face feature rows: 70,000/122,774
Name–face feature rows: 75,000/122,774
Name–face feature rows: 80,000/122,774
Name–face feature rows: 85,000/122,774
Name–face feature rows: 90,000/122,774
Name–face feature rows: 95,000/122,774
Name–face feature rows: 100,000/122,774
Name–face feature rows: 105,000/122,774
Name–face feature rows: 110,000/122,774
Name–face feature rows: 115,000/122,774
Name–face feature rows: 120,000/122,774
Name–face feature rows: 122,774/122,774
data/features/mm_nam

,split,group,rows,features,available_fraction,finite
0,train_sample,phrases,49943,14,1.000000,True
1,train_sample,names,49943,7,0.797029,True
2,validation,phrases,22126,14,1.000000,True
3,validation,names,22126,7,0.785501,True
4,test,phrases,50705,14,1.000000,True
5,test,names,50705,7,0.816310,True


## 4. Experiment Plan

The experiments test whether the **14 phrase–image features** and **seven name–face features** improve manipulation detection.

All comparisons use the same baseline: four whole-image–caption scores and four face–whole-caption scores from `05`.

| Step | Inputs added to the baseline | Total inputs | Question |
|---|---|---:|---|
| Baseline | None | 8 | What can the existing similarity scores do? |
| Phrase count control | Four phrase count/missing indicators | 12 | Do counts alone help? |
| Baseline + phrases | All 14 phrase features | 22 | Do phrase similarities add value beyond counts? |
| Name count control | Four name/face count and availability indicators | 12 | Does the presence of names/faces alone help? |
| Baseline + names | All seven name–face features | 15 | Do name–face matches add value beyond counts? |

All fits use the same training and validation rows, seed `4012`, and **StandardScaler + logistic regression** (`C=1`, `max_iter=2000`, no class reweighting). The scaler learns from training rows only. IDs, paths and manipulation annotations are excluded from the inputs.

Section 5 establishes baseline performance. Section 6 tests phrases and names separately, each against its count control. Section 7 records the differences and uncertainty. Whether to combine the groups is a follow-up decision based on these results.


In [30]:
# Join features by pair ID, preserving the same examples for every comparison.
joined = {}
for split in ["train_sample", "validation"]:
    rows = data[split]
    frame = rows.copy()
    for group_features in feature_groups.values():
        frame = frame.merge(group_features, on="row_id", how="left", validate="one_to_one", sort=False)
    assert frame.row_id.tolist() == rows.row_id.tolist()
    joined[split] = frame
train, val = joined["train_sample"], joined["validation"]
assert not (set(train.id) & set(val.id)), "Training/validation article overlap"

group_columns = {name: [c for c in frame if c != "row_id"] for name, frame in feature_groups.items()}
assert set(group_columns) == {"phrases", "names"}
controls = {
    "phrases": [c for c in group_columns["phrases"] if c.endswith(("_count", "_missing"))],
    "names": [c for c in group_columns["names"] if c.endswith(("_count", "_available"))],
}
BASE = CLIP_COLUMNS + FACE_COLUMNS
experiments = {
    "Baseline": BASE,
    "Phrase count control": BASE + controls["phrases"],
    "Baseline + phrases": BASE + group_columns["phrases"],
    "Name count control": BASE + controls["names"],
    "Baseline + names": BASE + group_columns["names"],
}
for columns in experiments.values():
    assert len(columns) == len(set(columns))
    assert np.isfinite(train[columns].to_numpy()).all() and np.isfinite(val[columns].to_numpy()).all()
genuine = val.is_fake.to_numpy() == 0
assert genuine.any()

## 5. Establish the Baseline

Each experiment fits two classifiers using the same selected features: one predicts **image manipulation**, the other **caption manipulation**. The existing helper combines their scores for **either modality manipulated**:

```text
overall_fake = 1 - (1 - image_fake_score) * (1 - caption_fake_score)
```

This is the same noisy-OR rule used in `10`; its probability interpretation assumes independence. The main tables below show **overall-fake detection**. All three outputs are evaluated; their detailed tables are available in section 7.

Two cutoff choices show different precision/recall trade-offs:

| View | How the cutoff is chosen | What to look at |
|---|---|---|
| **A. Balance precision and recall** | Choose the best validation F1 among tested cutoffs | Precision: how many flags are correct? Recall: how many manipulated pairs are caught? F1 combines both. |
| **B. Limit false alarms** | Allow at most 1% of genuine validation pairs to be flagged | How much manipulation can we catch while falsely flagging at most 10 per 1,000 genuine pairs? |

Genuine means **neither** image nor caption was manipulated. Ties are handled conservatively, so the false-positive rate can be below 1%. Both cutoffs are chosen using validation data, so the results describe this validation experiment.

The shared functions fit the classifiers, apply both cutoff rules and display the overall-detection results. The baseline run provides the reference for the feature comparisons.


In [31]:
def genuine_fpr_threshold(score, genuine, rate=0.01):
    values = np.asarray(score)[np.asarray(genuine, dtype=bool)]
    assert len(values) and np.isfinite(values).all() and 0 <= rate < 1
    allowed = int(np.floor(rate * len(values)))
    # >= threshold excludes all ties at the cutoff, so the allowed count is never exceeded.
    cutoff = np.sort(values)[len(values) - allowed - 1]
    return float(np.nextafter(cutoff, np.inf))


def fit_experiment(columns):
    assert len(columns) == len(set(columns))
    assert np.isfinite(train[columns].to_numpy()).all() and np.isfinite(val[columns].to_numpy()).all()
    predictions = {}
    for target in ["image_fake", "text_fake"]:
        assert train[target].nunique() == val[target].nunique() == 2
        classifier = make_pipeline(StandardScaler(), LogisticRegression(C=1., max_iter=2000, random_state=RANDOM_STATE))
        classifier.fit(train[columns], train[target])
        predictions[target] = classifier.predict_proba(val[columns])[:, 1]
    predictions["is_fake"] = combine_fake_score(predictions["image_fake"], predictions["text_fake"])
    return predictions


def run_experiment(name):
    """Fit this predeclared feature set; replacing its previous run avoids duplicate rows."""
    start = time.perf_counter()
    validation_predictions[name] = fit_experiment(experiments[name])
    training_seconds[name] = time.perf_counter() - start
    print(f"{name}: {len(experiments[name])} inputs, {training_seconds[name]:.1f}s", flush=True)


def evaluate_experiments():
    """Apply both cutoff rules to each of the three outputs, without refitting."""
    f1_rows, strict_rows = [], []
    for name, scores in validation_predictions.items():
        for target in TARGETS:
            score = scores[target]
            for records, threshold in [
                (f1_rows, best_f1_threshold(val[target], score)),
                (strict_rows, genuine_fpr_threshold(score, genuine)),
            ]:
                records.append({"experiment": name, "target": target, "n_features": len(experiments[name]),
                                **binary_metrics(val[target], score, threshold),
                                "genuine_FPR": float((score[genuine] >= threshold).mean())})
    f1_results, strict_results = pd.DataFrame(f1_rows), pd.DataFrame(strict_rows)
    assert (strict_results.genuine_FPR <= .01 + 1e-12).all()
    return f1_results, strict_results


def show_comparison(names):
    """Display only the overall target; retain all targets in the full result tables."""
    f1 = results[results.target == "is_fake"].set_index("experiment").loc[names].copy()
    strict = low_fpr_results[low_fpr_results.target == "is_fake"].set_index("experiment").loc[names].copy()
    baseline_f1 = results[(results.experiment == "Baseline") & (results.target == "is_fake")].iloc[0].F1
    baseline_recall = low_fpr_results[(low_fpr_results.experiment == "Baseline") &
                                     (low_fpr_results.target == "is_fake")].iloc[0].Recall
    f1["F1_change_vs_baseline"] = f1.F1 - baseline_f1
    strict["Recall_change_vs_baseline"] = strict.Recall - baseline_recall
    display(Markdown("**A. F1-selected cutoff — overall-fake detection**"))
    display(f1[["Precision", "Recall", "F1", "F1_change_vs_baseline", "genuine_FPR"]].round(4))
    display(Markdown("**B. At most 1% genuine false positives — overall-fake detection**"))
    display(strict[["genuine_FPR", "Precision", "Recall", "Recall_change_vs_baseline", "F1"]].round(4))

In [32]:
# Start a fresh evaluation run. This does not repeat feature extraction.
validation_predictions, training_seconds = {}, {}
run_experiment("Baseline")
results, low_fpr_results = evaluate_experiments()
show_comparison(["Baseline"])

Baseline: 8 inputs, 0.1s


**A. F1-selected cutoff — overall-fake detection**

,Precision,Recall,F1,F1_change_vs_baseline,genuine_FPR
experiment,,,,,
Baseline,0.6688,0.9896,0.7982,0.0,0.9611


**B. At most 1% genuine false positives — overall-fake detection**

,genuine_FPR,Precision,Recall,Recall_change_vs_baseline,F1
experiment,,,,,
Baseline,0.0099,0.9678,0.1518,0.0,0.2624


The F1-selected results show the balance between precision and recall, while the strict cutoff shows how much manipulation is caught with few genuine false positives. A high F1 can still coincide with many false alarms, so both tables matter.

`results` and `low_fpr_results` contain the cutoff values and full metrics for all three outputs. AUC measures score ranking and remains unchanged when the decision cutoff changes.


## 6. Test Each New Feature Group

### 6.1 Do phrase–image similarities help?

**Hypothesis:** similarities for individual caption phrases add information beyond the existing CLIP scores and the number of phrases.

The three fits use the baseline scores alone, add phrase counts/missing indicators, then add the full phrase group. The F1 and strict false-positive tables show the trade-offs at each step. Improvement over the count control is the comparison relevant to the added similarity measurements.


In [33]:
for name in ["Phrase count control", "Baseline + phrases"]:
    run_experiment(name)
results, low_fpr_results = evaluate_experiments()
show_comparison(["Baseline", "Phrase count control", "Baseline + phrases"])

Phrase count control: 12 inputs, 0.2s
Baseline + phrases: 22 inputs, 0.3s


**A. F1-selected cutoff — overall-fake detection**

,Precision,Recall,F1,F1_change_vs_baseline,genuine_FPR
experiment,,,,,
Baseline,0.6688,0.9896,0.7982,0.0000,0.9611
Phrase count control,0.6672,0.9923,0.7979,-0.0003,0.9707
Baseline + phrases,0.6671,0.9922,0.7978,-0.0003,0.9708


**B. At most 1% genuine false positives — overall-fake detection**

,genuine_FPR,Precision,Recall,Recall_change_vs_baseline,F1
experiment,,,,,
Baseline,0.0099,0.9678,0.1518,0.0000,0.2624
Phrase count control,0.0099,0.9683,0.1542,0.0025,0.2661
Baseline + phrases,0.0099,0.9720,0.1751,0.0233,0.2968


### 6.2 Do person-name–face matches help?

**Hypothesis:** the name–face matching measurements add information beyond the existing CLIP scores and the presence/number of names and faces.

The three fits use the baseline scores alone, add name/face counts and availability, then add the full name–face group. The same two cutoff views show whether the matching measurements contribute beyond those simpler controls. Similarity scores remain an imperfect indication of identity agreement.


In [34]:
for name in ["Name count control", "Baseline + names"]:
    run_experiment(name)
results, low_fpr_results = evaluate_experiments()
show_comparison(["Baseline", "Name count control", "Baseline + names"])

Name count control: 12 inputs, 0.2s
Baseline + names: 15 inputs, 0.4s


**A. F1-selected cutoff — overall-fake detection**

,Precision,Recall,F1,F1_change_vs_baseline,genuine_FPR
experiment,,,,,
Baseline,0.6688,0.9896,0.7982,0.0000,0.9611
Name count control,0.6671,0.9922,0.7978,-0.0003,0.9708
Baseline + names,0.6748,0.9782,0.7986,0.0005,0.9244


**B. At most 1% genuine false positives — overall-fake detection**

,genuine_FPR,Precision,Recall,Recall_change_vs_baseline,F1
experiment,,,,,
Baseline,0.0099,0.9678,0.1518,0.0000,0.2624
Name count control,0.0099,0.9709,0.1683,0.0165,0.2869
Baseline + names,0.0099,0.9716,0.1731,0.0213,0.2938


## 7. Results and Uncertainty

### 7.1 Are the ranking differences consistent across validation articles?

Compare each full group with baseline and its count control using **500 paired article-level bootstrap resamples**. Related pairs from one article stay together, and both models use the same resampled articles. Models are not retrained.

The table combines the measured F1 change, recall change at the strict cutoff, and AUC change with its 95% bootstrap interval. **Positive changes favour the added matching features.** An AUC interval spanning zero is inconclusive about the ranking gain. The interval covers AUC only; F1 and recall changes are point estimates. These exploratory comparisons are not adjusted for multiple testing.

The displayed comparison covers overall detection; the detailed tables include all three targets. Candidate features should be judged against their count controls, considering precision/recall trade-offs alongside AUC uncertainty. Combining the groups remains a possible follow-up experiment.


In [35]:
def prepare_weighted_auc(labels, scores):
    labels, scores = np.asarray(labels), np.asarray(scores)
    order = np.argsort(scores, kind="stable")
    starts = np.r_[0, np.flatnonzero(np.diff(scores[order])) + 1]
    return order, starts, labels[order].astype(float)


def weighted_auc(prepared, weights):
    order, starts, labels = prepared
    ordered = np.asarray(weights)[order]
    positive = np.add.reduceat(ordered * labels, starts)
    negative = np.add.reduceat(ordered * (1 - labels), starts)
    denominator = positive.sum() * negative.sum()
    if denominator == 0:
        return np.nan
    return float(np.sum(positive * (np.cumsum(negative) - 0.5 * negative)) / denominator)


assert set(validation_predictions) == set(experiments), "Run the baseline and both feature tests before this summary"
results, low_fpr_results = evaluate_experiments()
comparison_pairs = [
    ("Baseline + phrases", "Baseline", "Phrases vs baseline"),
    ("Baseline + phrases", "Phrase count control", "Phrases vs count control"),
    ("Baseline + names", "Baseline", "Names vs baseline"),
    ("Baseline + names", "Name count control", "Names vs count control"),
]
prepared = {(name, target): prepare_weighted_auc(val[target], score)
            for name, scores in validation_predictions.items() for target, score in scores.items()}
article_codes, articles = pd.factorize(val.id, sort=False)
rng = np.random.default_rng(RANDOM_STATE)
bootstrap_auc = {key: np.empty(BOOTSTRAPS) for key in prepared}
start = time.perf_counter()
for b in range(BOOTSTRAPS):
    multiplicity = np.bincount(rng.integers(0, len(articles), size=len(articles)), minlength=len(articles))
    weights = multiplicity[article_codes]
    for key, prepped in prepared.items():
        bootstrap_auc[key][b] = weighted_auc(prepped, weights)
    if (b + 1) % 100 == 0:
        print(f"Article bootstrap: {b + 1}/{BOOTSTRAPS}", flush=True)

f1_lookup = results.set_index(["experiment", "target"])
strict_lookup = low_fpr_results.set_index(["experiment", "target"])
difference_rows = []
for experiment, reference, label in comparison_pairs:
    for target in TARGETS:
        delta = bootstrap_auc[(experiment, target)] - bootstrap_auc[(reference, target)]
        valid = delta[np.isfinite(delta)]
        assert len(valid) >= .95 * BOOTSTRAPS, "Too few bootstrap samples contain both classes"
        lower, upper = np.quantile(valid, [.025, .975])
        difference_rows.append({"comparison": label, "experiment": experiment, "reference": reference, "target": target,
            "AUC_difference": float(f1_lookup.loc[(experiment, target), "AUC"] - f1_lookup.loc[(reference, target), "AUC"]),
            "CI_low": float(lower), "CI_high": float(upper), "valid_resamples": len(valid)})
auc_differences = pd.DataFrame(difference_rows)
findings = auc_differences.copy()
for metric in ["Precision", "Recall", "F1"]:
    findings[f"{metric}_change"] = [float(f1_lookup.loc[(r.experiment, r.target), metric] -
                                         f1_lookup.loc[(r.reference, r.target), metric]) for r in findings.itertuples()]
findings["recall_change_at_1pct_genuine_FPR"] = [float(
    strict_lookup.loc[(r.experiment, r.target), "Recall"] - strict_lookup.loc[(r.reference, r.target), "Recall"])
    for r in findings.itertuples()]
timings["article_bootstrap_seconds"] = time.perf_counter() - start
overall_findings = findings[findings.target == "is_fake"].set_index("comparison").copy()
overall_findings["AUC_95pct_interval"] = [f"[{r.CI_low:.4f}, {r.CI_high:.4f}]" for r in overall_findings.itertuples()]
display(overall_findings[["F1_change", "recall_change_at_1pct_genuine_FPR", "AUC_difference", "AUC_95pct_interval"]].round(4))

Article bootstrap: 100/500
Article bootstrap: 200/500
Article bootstrap: 300/500
Article bootstrap: 400/500
Article bootstrap: 500/500


,F1_change,recall_change_at_1pct_genuine_FPR,AUC_difference,AUC_95pct_interval
comparison,,,,
Phrases vs baseline,-0.0003,0.0233,0.0084,"[0.0068, 0.0100]"
Phrases vs count control,-0.0001,0.0209,0.0083,"[0.0068, 0.0098]"
Names vs baseline,0.0005,0.0213,0.0090,"[0.0073, 0.0106]"
Names vs count control,0.0008,0.0048,0.0034,"[0.0020, 0.0047]"


### 7.2 Detailed Results

Expand a table to inspect all three targets, manipulation types, feature coverage or training times. These tables stay in the notebook; no CSV or JSON result files are written. Save the notebook with its outputs to retain the results.


In [36]:
# Summarise both cutoff choices by manipulation type.
type_tables = []
for setting, metrics in [("validation_F1", results), ("validation_genuine_FPR_1pct", low_fpr_results)]:
    thresholds = metrics[metrics.target == "is_fake"].set_index("experiment").threshold
    for name, scores in validation_predictions.items():
        table = detection_rate_by_type(val, scores["is_fake"], float(thresholds.loc[name])).reset_index()
        table["experiment"], table["operating_point"] = name, setting
        type_tables.append(table)
by_type = pd.concat(type_tables, ignore_index=True)
runtimes = pd.DataFrame([{"experiment": name, "training_seconds": seconds}
                         for name, seconds in training_seconds.items()])


from IPython.display import HTML

detailed_tables = {
    "F1-selected metrics: all three targets": results,
    "Metrics at at most 1% genuine false positives": low_fpr_results,
    "Detection by manipulation type": by_type,
    "Paired AUC differences": auc_differences,
    "Feature comparison findings": findings,
    "Feature coverage": coverage,
    "Classifier training times": runtimes,
}
for title, table in detailed_tables.items():
    display(HTML(f"<details><summary>{title}</summary>{table.round(4).to_html(index=False)}</details>"))


experiment,target,n_features,AUC,PR-AUC,EER,ACC,Precision,Recall,F1,threshold,genuine_FPR
Baseline,image_fake,8,0.5822,0.5946,0.4390,0.5573,0.5509,0.9703,0.7028,0.3979,0.9157
Baseline,text_fake,8,0.8252,0.7169,0.2580,0.8095,0.6462,0.6215,0.6336,0.3667,0.1057
Baseline,is_fake,8,0.6800,0.8198,0.3819,0.6685,0.6688,0.9896,0.7982,0.4275,0.9611
Phrase count control,image_fake,12,0.5834,0.5954,0.4384,0.5530,0.5478,0.9801,0.7028,0.3838,0.9403
Phrase count control,text_fake,12,0.8260,0.7210,0.2545,0.8169,0.6744,0.5979,0.6338,0.3926,0.0890
Phrase count control,is_fake,12,0.6801,0.8203,0.3818,0.6670,0.6672,0.9923,0.7979,0.4175,0.9707
Baseline + phrases,image_fake,22,0.5873,0.5997,0.4392,0.5502,0.5459,0.9868,0.7030,0.3661,0.9560
Baseline + phrases,text_fake,22,0.8453,0.7522,0.2358,0.8240,0.6713,0.6584,0.6648,0.3661,0.1010
Baseline + phrases,is_fake,22,0.6884,0.8272,0.3767,0.6670,0.6671,0.9922,0.7978,0.4068,0.9708
Name count control,image_fake,12,0.5825,0.5957,0.4405,0.5508,0.5465,0.9828,0.7024,0.3784,0.9483


experiment,target,n_features,AUC,PR-AUC,EER,ACC,Precision,Recall,F1,threshold,genuine_FPR
Baseline,image_fake,8,0.5822,0.5946,0.4390,0.4658,0.6307,0.0230,0.0445,0.6703,0.0099
Baseline,text_fake,8,0.8252,0.7169,0.2580,0.8142,0.8909,0.3409,0.4932,0.6667,0.0099
Baseline,is_fake,8,0.6800,0.8198,0.3819,0.4349,0.9678,0.1518,0.2624,0.8608,0.0099
Phrase count control,image_fake,12,0.5834,0.5954,0.4384,0.4651,0.6234,0.0209,0.0405,0.6736,0.0099
Phrase count control,text_fake,12,0.8260,0.7210,0.2545,0.8177,0.8976,0.3527,0.5064,0.6596,0.0099
Phrase count control,is_fake,12,0.6801,0.8203,0.3818,0.4365,0.9683,0.1542,0.2661,0.8589,0.0099
Baseline + phrases,image_fake,22,0.5873,0.5997,0.4392,0.4657,0.6434,0.0209,0.0404,0.6799,0.0099
Baseline + phrases,text_fake,22,0.8453,0.7522,0.2358,0.8311,0.9006,0.4079,0.5615,0.6386,0.0099
Baseline + phrases,is_fake,22,0.6884,0.8272,0.3767,0.4504,0.9720,0.1751,0.2968,0.8506,0.0099
Name count control,image_fake,12,0.5825,0.5957,0.4405,0.4659,0.6355,0.0228,0.0440,0.6734,0.0099


fake_cls,flagged_rate,n,experiment,operating_point
orig,0.9611,7473,Baseline,validation_F1
text_attribute,0.9758,825,Baseline,validation_F1
face_attribute,0.9820,3999,Baseline,validation_F1
face_swap,0.9893,4788,Baseline,validation_F1
face_attribute&text_attribute,0.9918,487,Baseline,validation_F1
face_swap&text_attribute,0.9944,533,Baseline,validation_F1
text_swap,0.9989,1894,Baseline,validation_F1
face_attribute&text_swap,1.0000,964,Baseline,validation_F1
face_swap&text_swap,1.0000,1163,Baseline,validation_F1
orig,0.9707,7473,Phrase count control,validation_F1


comparison,experiment,reference,target,AUC_difference,CI_low,CI_high,valid_resamples
Phrases vs baseline,Baseline + phrases,Baseline,image_fake,0.0051,0.0036,0.0068,500
Phrases vs baseline,Baseline + phrases,Baseline,text_fake,0.0201,0.0175,0.0228,500
Phrases vs baseline,Baseline + phrases,Baseline,is_fake,0.0084,0.0068,0.0100,500
Phrases vs count control,Baseline + phrases,Phrase count control,image_fake,0.0039,0.0025,0.0054,500
Phrases vs count control,Baseline + phrases,Phrase count control,text_fake,0.0193,0.0168,0.0220,500
Phrases vs count control,Baseline + phrases,Phrase count control,is_fake,0.0083,0.0068,0.0098,500
Names vs baseline,Baseline + names,Baseline,image_fake,0.0102,0.0072,0.0131,500
Names vs baseline,Baseline + names,Baseline,text_fake,0.0063,0.0038,0.0089,500
Names vs baseline,Baseline + names,Baseline,is_fake,0.0090,0.0073,0.0106,500
Names vs count control,Baseline + names,Name count control,image_fake,0.0099,0.0070,0.0129,500


comparison,experiment,reference,target,AUC_difference,CI_low,CI_high,valid_resamples,Precision_change,Recall_change,F1_change,recall_change_at_1pct_genuine_FPR
Phrases vs baseline,Baseline + phrases,Baseline,image_fake,0.0051,0.0036,0.0068,500,-0.0049,0.0166,0.0002,-0.0022
Phrases vs baseline,Baseline + phrases,Baseline,text_fake,0.0201,0.0175,0.0228,500,0.0251,0.0368,0.0311,0.0670
Phrases vs baseline,Baseline + phrases,Baseline,is_fake,0.0084,0.0068,0.0100,500,-0.0017,0.0026,-0.0003,0.0233
Phrases vs count control,Baseline + phrases,Phrase count control,image_fake,0.0039,0.0025,0.0054,500,-0.0019,0.0067,0.0001,-0.0001
Phrases vs count control,Baseline + phrases,Phrase count control,text_fake,0.0193,0.0168,0.0220,500,-0.0031,0.0605,0.0309,0.0552
Phrases vs count control,Baseline + phrases,Phrase count control,is_fake,0.0083,0.0068,0.0098,500,-0.0000,-0.0001,-0.0001,0.0209
Names vs baseline,Baseline + names,Baseline,image_fake,0.0102,0.0072,0.0131,500,-0.0020,0.0118,0.0015,-0.0014
Names vs baseline,Baseline + names,Baseline,text_fake,0.0063,0.0038,0.0089,500,0.0382,-0.0019,0.0168,0.0540
Names vs baseline,Baseline + names,Baseline,is_fake,0.0090,0.0073,0.0106,500,0.0060,-0.0115,0.0005,0.0213
Names vs count control,Baseline + names,Name count control,image_fake,0.0099,0.0070,0.0129,500,0.0024,-0.0008,0.0018,-0.0012


split,group,rows,features,available_fraction,finite
train_sample,phrases,49943,14,1.0000,True
train_sample,names,49943,7,0.7970,True
validation,phrases,22126,14,1.0000,True
validation,names,22126,7,0.7855,True
test,phrases,50705,14,1.0000,True
test,names,50705,7,0.8163,True


experiment,training_seconds
Baseline,0.1128
Phrase count control,0.1963
Baseline + phrases,0.2878
Name count control,0.2113
Baseline + names,0.4163


# Key Findings

**Phrase–image features are worth further exploration.** With genuine false positives capped at 1%, overall recall increased from 15.18% for baseline to 17.51% with phrases. The phrase count control reached 15.42%, so the matching features contributed a further **2.09 percentage points**. The clearest benefit was in caption-manipulation detection: F1 increased from 0.6336 to 0.6648. Overall AUC also improved beyond the count control, although overall F1 was essentially unchanged. This supports further testing of phrases, particularly for caption-related manipulation.

**Name–face matching is a lower priority for overall detection.** Strict overall recall was 17.31%, compared with 16.83% for the name/face count control—an additional **0.48 percentage points**. Overall AUC showed a small positive gain beyond counts, and image-manipulation ranking also improved. However, matching added little to caption detection and did not improve strict image-manipulation recall. Its contribution is limited in this setup.

**Absolute detection performance remains limited.** The F1-selected baseline flagged 96.11% of genuine pairs. Restricting genuine false positives to 1% leaves all variants catching fewer than 18% of manipulated pairs. The modest ranking and recall gains therefore do not yet produce a strong overall detector.